# DEGs of All Cell Types

In [ ]:
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
import pandas as pd
from Xenium_5K_IPF_constants i

# Constant Declaration
deg_num = 500

# Option
sc.settings.verbosity = 3
plt.rcParams["font.family"] = "Fira Sans" 
%matplotlib ipympl

In [ ]:
adata = sc.read_h5ad(f"{WORK_PATH}AnnData/integrated_anndata.h5ad")
celltype_df=pd.read_csv(f"{WORK_PATH}CSV/013_merge_type.csv", index_col="cell_id")

In [ ]:
adata = adata[celltype_df.index,].copy()

In [ ]:
adata.obs = pd.merge(adata.obs, celltype_df[["cell_type", "cell_type_major", "subtype",]], left_index=True, right_index=True)

sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [ ]:
adata.obs["subtype"]=pd.Categorical(adata.obs["subtype"], categories=ALL_SUBTYPE_LIST)

In [ ]:
sc.tl.dendrogram(adata, groupby="subtype")
sc.tl.rank_genes_groups(adata, groupby="subtype", use_raw=False, n_genes=adata.shape[1], method='wilcoxon')

In [ ]:
rg = adata.uns["rank_genes_groups"]
groups = list(rg["names"].dtype.names)

gene_to_col = {g: i for i, g in enumerate(adata.var_names)}

# fraction array by each group
frac_by_group = {}
obs_grp = adata.obs["subtype"].values

for g in groups:
    mask = (obs_grp == g)
    n_cells = int(mask.sum())
    if n_cells == 0:
        frac_by_group[g] = np.zeros(adata.n_vars, dtype=float)
        continue

    subX = adata.X[mask, :].tocsr()
    hits = subX.copy()
    hits.data = (hits.data > 0).astype(np.int8)
    frac = np.asarray(hits.sum(axis=0)).ravel() / n_cells

    frac_by_group[g] = frac

In [ ]:
subtype_deg_df = pd.DataFrame(columns=[i for i in range(deg_num*len(groups))], index=["subtype", "genes", "fractions", 'scores', 'pvals', 'pvals_adj', 'logfoldchanges'])

for n, g in enumerate(groups):
    for i in range(deg_num):
        subtype_deg_df.loc["subtype", i+deg_num*n]=g
        subtype_deg_df.loc["genes", i+deg_num*n]=rg["names"][i][n]
        subtype_deg_df.loc["fractions", i+deg_num*n]=frac_by_group[g][gene_to_col[rg["names"][i][n]]]
        for t in ['scores', 'pvals', 'pvals_adj', 'logfoldchanges']:
            subtype_deg_df.loc[t, i+deg_num*n]=rg[t][i][n].astype(np.float64)
subtype_df = subtype_deg_df.T
subtype_df.to_csv(f"{WORK_PATH}CSV/015_all_cell_type_degs.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Collect DEGs without duplication
dedup_deg = {g:[] for g in groups}
all_degs = []
for g in groups:
    hoge = 0
    temp_df = subtype_df[subtype_df["subtype"]==g]
    temp_list = list(temp_df["genes"])
    for l in temp_list:
        if hoge >= 2:
            continue
        if l not in all_degs:
            temp_df_g = temp_df[temp_df["genes"]==l]
            if (temp_df_g.loc[:,"fractions"].values[0] > 0.1) & (temp_df_g.loc[:,"logfoldchanges"].values[0] > 1.5) :
                dedup_deg[g].append(l)
                all_degs.append(l)
                hoge += 1            

# Change the position of FRC and FDC
for g in dedup_deg["FRC"]+dedup_deg["FDC"]:
    all_degs.remove(g)
all_degs[all_degs.index(dedup_deg["Venule"][-1])+1:all_degs.index(dedup_deg["Venule"][-1])+1]=dedup_deg["FRC"]+dedup_deg["FDC"]

In [ ]:
# Dataframe for dotplot
dot_deg_df = pd.DataFrame(columns=[i for i in range(len(all_degs)*len(groups))], index=['3m', "subtype", "genes", "fractions", 'logfoldchanges'])
for n, g in enumerate(groups):
    group_deg_list = [rg["names"][m][n] for m in range(len(adata.var_names))]
    for i, d in enumerate(all_degs):
        deg_idx = [n for n, dg in enumerate(group_deg_list) if dg == d][0]
        dot_deg_df.loc["3m", i+len(all_degs)*n]=CELL_TYPE_3M[adata[adata.obs["subtype"]==g].obs["cell_type_major"].iloc[0]]
        dot_deg_df.loc["subtype", i+len(all_degs)*n]=g
        dot_deg_df.loc["genes", i+len(all_degs)*n]=d
        dot_deg_df.loc["fractions", i+len(all_degs)*n]=frac_by_group[g][gene_to_col[d]]
        dot_deg_df.loc['logfoldchanges', i+len(all_degs)*n]=rg['logfoldchanges'][deg_idx][n].astype(np.float64)
tp_3m = ["Epithelial", "Stromal", "Immune"]
dot_deg_df_3m=[dot_deg_df[[n for n,bl in enumerate(dot_deg_df.loc["3m"]==tp_3) if bl]] for tp_3 in tp_3m]

In [ ]:
# Dotplot
wid_r = [int(ddf3.shape[1]/len(all_degs)) for ddf3 in dot_deg_df_3m]
fig, axes = plt.subplots(1,3,figsize=(12, 15), width_ratios=wid_r)
fig.subplots_adjust(left=0.08, top=0.97, right=0.9, bottom=0.13, wspace=0.02)

for N_g, ax, tp_3, ddf3 in zip(wid_r, axes, tp_3m, dot_deg_df_3m):
    
    N_d = len(all_degs)

    # X axis of cell types
    x = np.repeat(np.arange(N_g), N_d)
    
    # Y axies of genes
    y = np.tile(np.arange(N_d), N_g)
    flat_idx = [i + N_d*n for n in range(N_g) for i in range(N_d)]
    
    # Dot size represents the fractions
    size_scale = 100
    sizes  = np.asarray(ddf3.loc["fractions"])[flat_idx].astype(float) * size_scale
    colors = np.asarray(ddf3.loc["logfoldchanges"])[flat_idx].astype(float)
    
    # plot
    sc_fig = ax.scatter(
        x, y,
        s=sizes,
        c=colors,
        cmap="bwr",
        vmin=-7, vmax=7,
        edgecolors="none"
    )

    ax.set_xticks(range(N_g))
    ax.set_xticklabels(ddf3.loc["subtype"].unique(), rotation=60, rotation_mode="anchor", va="top", ha="right", fontsize=12)
    ax.set_yticks(range(N_d), labels=[])
    ax.set_xlim(-0.5, N_g-0.5)
    ax.set_ylim(N_d, -1)  # 
    ax.set_title(tp_3, fontsize=13)
    
axes[0].set_yticklabels(all_degs, fontsize=12)

for n in [1,2]:
    axes[n].tick_params("y", size=0)

# colorbar legend (log2FC)
cax = fig.add_axes([0.92, 0.6, 0.02, 0.12])
cbar = fig.colorbar(sc_fig, cax=cax)
cbar.set_label("$\log_{2}{(\mathrm{fold \; change})}$", fontsize=12)

# dot size legend
dax = fig.add_axes([0.92, 0.4, 0.02, 0.12])
dax.yaxis.set_label_position("right")
dax.yaxis.set_ticks_position("right")
dax.set_ylabel("Fraction of cells in group (%)", fontsize=12)
dax.scatter(
    [1]*5, [0,1,2,3,4],
    s=[n*size_scale for n in [0.2,0.4,0.6,0.8,1]],
    c="grey",
    edgecolors="none"
)
dax.set_ylim(-0.5, 4.5)
dax.set_yticks([0,1,2,3,4], labels=["20","40","60","80","100"])
dax.set_xticks([1], labels=[])
dax.tick_params("x", size=0)
for dd in ["left", "right", "top", "bottom"]:
    dax.spines[dd].set_visible(False)

plt.savefig(f"{WORK_PATH}Figure/015_all_cell_type_degs_dotplot.png", dpi=500, format="png", bbox_inches=None)
plt.show()

# Counting by Cell Type

In [ ]:
import pandas as pd
import numpy as np
from Xenium_5K_IPF_constants import CELL_TYPE_3M
import matplotlib.pyplot as plt
from Xenium_5K_IPF_constants import SUBTYPE_HATCHES, WORK_PATH

In [ ]:
celltype_df=pd.read_csv(f"{WORK_PATH}CSV/013_merge_type.csv", index_col="cell_id")
celltype_df["3m"] = celltype_df["cell_type_major"].map(CELL_TYPE_3M)
tp_3m = ["Epithelial", "Stromal", "Immune"]
m_df = [celltype_df[celltype_df["3m"]==tp_3].value_counts("subtype") for tp_3 in tp_3m]

In [ ]:
df = pd.crosstab(celltype_df["subtype"], celltype_df["sample"])
df_rt = 100*df/df.sum()
df_rt=df_rt.reindex(index=list(SUBTYPE_HATCHES))
df_rt.to_csv(f"{WORK_PATH}CSV/015_alltype_counts_by_sample.csv", encoding="utf-8-sig")

In [ ]:
fig, axes = plt.subplots(1,3, width_ratios=[len(m_df[n]) for n in [0,1,2]], figsize=(8.5,3.2))
plt.subplots_adjust(left=0.05, right=0.99, bottom=0.47, top=0.9, wspace=0.15)
[m_df[n].plot.bar(ax=axes[n], color=c) for n,c in enumerate(["#ff4b00", "#03af7a", "#005aff"])]
[ax.set_xlabel("") for ax in axes]
[ax.set_title(t, weight=500, fontsize=14) for ax,t in zip(axes,["Epithelial", "Stromal", "Immune"])]

vmax = [m.max() for m in m_df]

for ax, vm in zip(axes, vmax):
    v_lim = ((vm//20e3)+1)*20e3 
    ax.set_ylim(0,v_lim)
    yticks = np.linspace(0, v_lim, 5)
    ax.set_yticks(ticks=[y for y in yticks], labels=[f"{int(y/1000)}K" for y in yticks], fontsize=9)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=60, rotation_mode="anchor", va="top", ha="right", fontsize=10)

plt.savefig(f"{WORK_PATH}Figure/015_all_celltype_counts.png", dpi=500, format="png", bbox_inches=None)
plt.show()

# Neighborfood Ennrichment Analysis by Squidpy

In [ ]:
import scanpy as sc
import squidpy as sq
from scipy.spatial.distance import pdist
from scipy.cluster.hierarchy import linkage, leaves_list, dendrogram, set_link_color_palette
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import scicm
import pandas as pd
from Xenium_5K_IPF_constants import ALL_SUBTYPE_LIST, WORK_PATH

# Option
sc.settings.verbosity = 3
plt.rcParams["font.family"] = "Fira Sans" 
%matplotlib ipympl

In [ ]:
celltype_df=pd.read_csv(f"{WORK_PATH}CSV/013_merge_type.csv", index_col="cell_id")
adata = sc.read_h5ad(f"{WORK_PATH}AnnData/spatial_merged.h5ad")

In [ ]:
adata = adata[celltype_df.index, ].copy()
adata.obs["subtype"]=celltype_df["subtype"]
adata.obs["subtype"]=pd.Categorical(values=adata.obs["subtype"], categories=ALL_SUBTYPE_LIST, ordered=True)

In [ ]:
sq.gr.spatial_neighbors(adata, coord_type="generic", delaunay=True)
sq.gr.nhood_enrichment(adata, cluster_key="subtype")

In [ ]:
# Customize heatmap
Z = pd.DataFrame(adata.uns["subtype_nhood_enrichment"]["zscore"], index=ALL_SUBTYPE_LIST, columns=ALL_SUBTYPE_LIST)

# hierarchical clustering
D = pdist(Z.values, metric="correlation")
L = linkage(D, method="average")

# label sort
ordered_labels = Z.index[leaves_list(L)]
Z_reordered = Z.loc[ordered_labels, ordered_labels]
v_list = [Z_reordered.iloc[m, m:Z_reordered.shape[0]].to_list() for m in range(len(ALL_SUBTYPE_LIST))]

In [ ]:
margin=[0.01, 0.01, 0.75, 0.99]
fig_wd = 10
dd_rt = 1.2

fig, (ax1, ax2) = plt.subplots(2,1,figsize=(fig_wd, fig_wd*(margin[2]-margin[0])/(margin[3]-margin[1])*10/(10-dd_rt)), height_ratios=[dd_rt,10-dd_rt])
fig.subplots_adjust(left=margin[0], bottom=margin[1], right=margin[2], top=margin[3], hspace=0)

cmap = plt.get_cmap('tab10')
set_link_color_palette([matplotlib.colors.to_hex(cmap(n)) for n in range(10)])
dd_plot = dendrogram(
    L,
    orientation="top",
    no_labels=True,
    ax=ax1,
    color_threshold=0.8,
    above_threshold_color="#666666",
)

line_c = [c for c in ax1.get_children() if isinstance(c, matplotlib.collections.LineCollection)]
for l_c in line_c:
    l_c.set(linewidth=1.8)
    
bx = [1/len(ALL_SUBTYPE_LIST), 1/len(ALL_SUBTYPE_LIST)]  # matrix size

norm = matplotlib.colors.Normalize(vmin=0, vmax=60)
sm = plt.cm.ScalarMappable(cmap="scicm.BgreyY", norm=norm)

for m, l in enumerate(v_list):
    for n, t in enumerate(l):
        r = matplotlib.patches.Rectangle(xy=((n+m)*bx[0], 1-(m+1)*bx[1]) , width=bx[0], height=bx[1], fc=sm.to_rgba(t, bytes=False), ec="#777777", lw=0.5)
        ax2.add_patch(r)

ax1.axis("off")
ax2.spines["left"].set_visible(False)
ax2.spines["bottom"].set_visible(False)
ax2.set_yticks([(n+1/2)*(1/len(ALL_SUBTYPE_LIST)) for n in range(len(ALL_SUBTYPE_LIST))], labels=ordered_labels[::-1], fontsize=fig_wd*1.5)        
ax2.yaxis.set_ticks_position('right')
ax2.set_xticks([])
ax2.set_xlim(0,1)
ax2.set_ylim(0,1)

cbar_ax = fig.add_axes([margin[0], 0.45, 0.3, 0.02])
cbar = fig.colorbar(sm, cax=cbar_ax,  orientation="horizontal", extend="both")
cbar.outline.set_linewidth(0.5)
cbar.ax.tick_params("x", labelsize=fig_wd*1.4)
cbar.ax.set_title("Neighbor enrichment z-score", fontsize=fig_wd*1.5)
cbar_ax.set_xticks([0,30,60])

plt.show()
fig.savefig(f"{WORK_PATH}Figure/015_neighbor_enrichment_score.png", dpi=500, format="png", bbox_inches=None)

# Enrichment Analysis by Each Subtype

In [ ]:
import scanpy as sc
import decoupler as dc
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from Xenium_5K_IPF_constants import ALL_SUBTYPE_LIST, WORK_PATH

# Option
sc.settings.verbosity = 3
plt.rcParams["font.family"] = "Fira Sans" 
%matplotlib ipympl

In [ ]:
adata = sc.read_h5ad(f"{WORK_PATH}AnnData/integrated_anndata.h5ad")
celltype_df=pd.read_csv(f"{WORK_PATH}CSV/013_merge_type.csv", index_col="cell_id")

In [ ]:
adata.obs = pd.merge(adata.obs, celltype_df[["subtype", "cell_type_major"]], left_index=True, right_index=True, how="left")

In [ ]:
progeny = dc.op.progeny(organism="human")

In [ ]:
dc.mt.mlm(data=adata, net=progeny)
score = dc.pp.get_obsm(adata=adata, key="score_mlm")

In [ ]:
df = score.to_df()
df["subtype"]=score.obs["subtype"]
cluster_df = df.groupby("subtype").mean()
cluster_df = cluster_df.loc[ALL_SUBTYPE_LIST]

In [ ]:
sns.clustermap(
    data=cluster_df,
    z_score=1,
    dendrogram_ratio=(0,0.05),
    method="average",
    row_cluster=False,
    cmap="PuOr_r",
    figsize=(5.5,10),
    vmax=4, vmin=-2.5,
    cbar_kws={"orientation":"horizontal", "ticks":[-2, 0 ,2, 4]}
)
plt.subplots_adjust(right=0.6, bottom=0.18)

chi = plt.gcf().get_children()
chi[3].tick_params("y", labelsize=13)
chi[3].tick_params("x", labelsize=13)
chi[3].set_xlabel("")
chi[3].set_ylabel("")
chi[4].set_position((0.7, 0.072, 0.2, 0.025))
chi[4].tick_params("x", labelsize=12)
chi[4].set_title("Z-score\n(for pathways)", fontsize=13)

for ch in [chi[1], chi[2]]:
    line_c = [c for c in ch.get_children() if isinstance(c, matplotlib.collections.LineCollection)]
    for l_c in line_c:
        l_c.set(linewidth=1)
    
plt.savefig(f"{WORK_PATH}Figure/015_subtype_enrichment_Progeny.png", dpi=500, format="png", bbox_inches=None)